# 01 — Problem and Data (Buổi 1)

**Đề tài:** Phân tích bán hàng, quản lý tồn kho và phân nhóm khách hàng
**Cửa hàng:** Nhà sách và văn phòng phẩm
**Hình thức:** Thực hiện 1 mình

Notebook này mô tả bài toán, câu hỏi nghiên cứu, từ điển dữ liệu và kiểm tra dữ liệu mẫu GV.


## 1. Bài toán

Xây pipeline KHDL cho cửa hàng nhà sách – văn phòng phẩm:
1. Quản lý tồn kho OOP có nhật ký
2. Phân tích doanh thu / sản phẩm / cặp mua cùng
3. Phân nhóm khách hàng RFM
4. Dự đoán giá trị đơn hàng (và cảnh báo tồn kho thấp)

### Câu hỏi nghiên cứu
1. Danh mục và kênh bán nào tạo doanh thu lớn nhất?
2. Sản phẩm nào bán chạy/bán chậm; cặp nào mua cùng nhau?
3. Thời điểm nào doanh thu cao nhất?
4. Khách hàng chia thành những nhóm RFM nào?
5. Dự đoán giá trị đơn hàng đạt sai số bao nhiêu?
6. Sản phẩm nào nguy cơ dưới reorder_level?


## 2. Nguồn dữ liệu

| Nguồn | Vai trò |
|---|---|
| products_lecturer (60 SP) | Khởi đầu do GV cung cấp |
| DummyJSON Products API | Crawl bổ sung |
| Books to Scrape | Crawl bổ sung |
| HTML mẫu / Mock API | Thực hành (không tính SP mới) |

Mục tiêu: >= 80 sản phẩm sau khi gộp.


In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path('..').resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

RAW = ROOT / 'data' / 'raw'
print('ROOT:', ROOT)
print('Raw files:', sorted(p.name for p in RAW.glob('*')))


ROOT: C:\Cao Hoc\Lập trình KHDL\Chuyên đề\NOP_GIAO_VIEN_CHUYEN_DE_6
Raw files: ['products_crawled.csv', 'products_from_html_practice.csv', 'products_lecturer.csv', 'products_lecturer.xlsx', 'products_sample.csv', 'products_sample.json', 'products_sample.xlsx']


## 3. Đọc dữ liệu mẫu giảng viên

In [2]:
df = pd.read_csv(RAW / 'products_lecturer.csv')
print(df.shape)
df.head()


(60, 12)


,product_id,product_name,category,brand,unit,unit_price,initial_quantity,reorder_level,popularity_weight,paired_product_id,source_type,source_reference
0,P0001,Vở kẻ ngang 80 trang,Vở và giấy,Campus,quyển,12000,180,35,1,P0011,lecturer_sample,Bộ dữ liệu mẫu Chuyên đề 6
1,P0002,Vở kẻ ngang 120 trang,Vở và giấy,Campus,quyển,18000,150,30,8,P0011,lecturer_sample,Bộ dữ liệu mẫu Chuyên đề 6
2,P0003,Vở ô ly 96 trang,Vở và giấy,Hồng Hà,quyển,14000,170,35,5,P0017,lecturer_sample,Bộ dữ liệu mẫu Chuyên đề 6
3,P0004,Sổ lò xo A5 200 trang,Vở và giấy,Hải Tiến,quyển,42000,90,20,2,NaN,lecturer_sample,Bộ dữ liệu mẫu Chuyên đề 6
4,P0005,Sổ tay bìa da A5,Vở và giấy,OEM,quyển,79000,55,15,9,NaN,lecturer_sample,Bộ dữ liệu mẫu Chuyên đề 6


## 4. Từ điển dữ liệu (products)

In [3]:
data_dict = pd.DataFrame([
    {'field': 'product_id', 'type': 'string', 'meaning': 'Mã sản phẩm (PK)'},
    {'field': 'product_name', 'type': 'string', 'meaning': 'Tên sản phẩm'},
    {'field': 'category', 'type': 'string', 'meaning': 'Danh mục'},
    {'field': 'brand', 'type': 'string', 'meaning': 'Thương hiệu'},
    {'field': 'unit', 'type': 'string', 'meaning': 'Đơn vị bán'},
    {'field': 'unit_price', 'type': 'float', 'meaning': 'Giá niêm yết'},
    {'field': 'initial_quantity', 'type': 'int', 'meaning': 'Tồn kho ban đầu'},
    {'field': 'reorder_level', 'type': 'int', 'meaning': 'Mức cảnh báo tồn kho'},
    {'field': 'popularity_weight', 'type': 'float', 'meaning': 'Trọng số mô phỏng khả năng mua'},
    {'field': 'paired_product_id', 'type': 'string', 'meaning': 'SP thường mua cùng (mô phỏng)'},
    {'field': 'source_type', 'type': 'string', 'meaning': 'Loại nguồn dữ liệu'},
    {'field': 'source_reference', 'type': 'string', 'meaning': 'Mô tả/URL nguồn'},
])
data_dict


,field,type,meaning
0,product_id,string,Mã sản phẩm (PK)
1,product_name,string,Tên sản phẩm
2,category,string,Danh mục
3,brand,string,Thương hiệu
4,unit,string,Đơn vị bán
5,unit_price,float,Giá niêm yết
6,initial_quantity,int,Tồn kho ban đầu
7,reorder_level,int,Mức cảnh báo tồn kho
8,popularity_weight,float,Trọng số mô phỏng khả năng mua
9,paired_product_id,string,SP thường mua cùng (mô phỏng)


## 5. Kiểm tra chất lượng nhanh

In [4]:
print('Missing per column:')
print(df.isna().sum())
print('\nDuplicate product_id:', df['product_id'].duplicated().sum())
print('unit_price <= 0:', (df['unit_price'] <= 0).sum())
print('initial_quantity <= reorder_level:', (df['initial_quantity'] <= df['reorder_level']).sum())
print('\nCategories:')
print(df['category'].value_counts())
print('\nPrice describe:')
print(df['unit_price'].describe())


Missing per column:
product_id            0
product_name          0
category              0
brand                 0
unit                  0
unit_price            0
initial_quantity      0
reorder_level         0
popularity_weight     0
paired_product_id    45
source_type           0
source_reference      0
dtype: int64

Duplicate product_id: 0
unit_price <= 0: 0
initial_quantity <= reorder_level: 0

Categories:
category
Vở và giấy            10
Bút viết              10
Dụng cụ học tập       10
Hồ sơ và lưu trữ      10
Thiết bị văn phòng    10
Phụ kiện máy tính     10
Name: count, dtype: int64

Price describe:
count    6.000000e+01
mean     2.076333e+05
std      4.901136e+05
min      4.000000e+03
25%      1.750000e+04
50%      6.000000e+04
75%      1.687500e+05
max      3.190000e+06
Name: unit_price, dtype: float64


## 6. Bảng dữ liệu dự kiến (sẽ sinh / crawl ở buổi sau)

- products (>= 80)
- customers (>= 200)
- orders (>= 1000)
- order_details (>= 3000)
- inventory_transactions

Xem sơ đồ ER trong reports/01_hoso_buoi_1.md.
